# Fase 2 — Início da revisão persistente
Este notebook usa as fixtures fictícias e os mesmos serviços do terminal, sem API/LLM.
Aceitar confirma a interpretação; não cria projeto, pagamento ou compromisso.
Demonstra autoria, correção, fontes imutáveis, versão e persistência. Agente/ferramentas/APIs ainda serão implementados.

In [1]:
# CONTEXTO: identidade de demonstração é entrada do serviço, não um nome extraído pelo modelo.
from pathlib import Path
import sys, tempfile
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/secretaria').is_dir())
sys.path.insert(0,str(raiz/'src'))
from secretaria.banco import Banco,Contexto
from secretaria.extracao import analisar_demonstracao
from secretaria.fixtures_extracao import criar_modelo_fixture
from secretaria.fila_revisao import FilaRevisao
temporario=tempfile.TemporaryDirectory()
caminho=Path(temporario.name)/'revisao.sqlite3'
banco=Banco(caminho);banco.preparar_demo()
ctx=Contexto('demo','ana')
print('Demonstração local: banco temporário, sem API.')

Demonstração local: banco temporário, sem API.


In [2]:
# CHAIN/FILA: extração manual identificada produz propostas; fila revalida suas fontes.
resultado=analisar_demonstracao(banco,ctx,criar_modelo_fixture(),'fixture_manual')
fila=FilaRevisao(banco)
cadastro=fila.registrar_resultado(ctx,resultado)
assert cadastro['novas']==8
assert fila.registrar_resultado(ctx,resultado)['novas']==0
propostas=fila.listar(ctx)
print('Interpretações pendentes:',len(propostas))

Interpretações pendentes: 8


In [3]:
# REVISÃO: corrige interpretação, mantém fontes e confirma a versão efetivamente exibida.
primeira=propostas[0]
corrigida=fila.corrigir(ctx,primeira['id'],primeira['versao'],
    {'resumo':'Implantação do portal de atendimento da Alfa.'},'Resumo conferido neste exercício.')
assert corrigida['dados']['fontes']==primeira['dados']['fontes']
aceita=fila.decidir(ctx,corrigida['id'],corrigida['versao'],'aceitar','Conferi a interpretação e as fontes.')
assert aceita['estado']=='aceita'
assert fila.decidir(ctx,aceita['id'],corrigida['versao'],'aceitar','Repetição')==aceita
segunda=propostas[1]
rejeitada=fila.decidir(ctx,segunda['id'],segunda['versao'],'rejeitar','Exemplo didático de rejeição humana.')
print('Decisões:',aceita['estado'],rejeitada['estado'])

Decisões: aceita rejeitada


In [4]:
# VERSÃO: uma tela antiga não pode confirmar conteúdo corrigido por outra operação.
terceira=propostas[2]
fila.corrigir(ctx,terceira['id'],terceira['versao'],{'resumo':'Entrada prevista, sem comprovação de pagamento.'},'Esclarecer previsão.')
try:
    fila.decidir(ctx,terceira['id'],terceira['versao'],'aceitar','Tela antiga')
    raise AssertionError('Deveria bloquear a versão antiga.')
except ValueError as erro:
    print('Bloqueio esperado:',erro)
try:
    fila.obter(Contexto('demo','bruno'),primeira['id'])
    raise AssertionError('Deveria bloquear outro usuário.')
except PermissionError:
    print('Fila pessoal de outro usuário bloqueada.')

Bloqueio esperado: Versão mudou. Leia a proposta atual antes de decidir.
Fila pessoal de outro usuário bloqueada.


In [5]:
# PERSISTÊNCIA/AUDITORIA: reabre SQLite e confere decisões e antes/depois.
banco.fechar()
banco=Banco(caminho);fila=FilaRevisao(banco)
assert fila.obter(ctx,primeira['id'])['estado']=='aceita'
assert fila.obter(ctx,segunda['id'])['estado']=='rejeitada'
historico=fila.historico(ctx,primeira['id'])
assert [h['acao'] for h in historico]==['registrar','corrigir','aceitar']
assert fila.registrar_resultado(ctx,resultado)['novas']==0
# As tabelas operacionais existem desde as migrações das fases seguintes; aceitar não pode preenchê-las.
operacionais={t:banco.con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
              for t in ('projetos','orcamentos','pagamentos','agendamentos')}
assert set(operacionais.values())=={0}, operacionais
print('Histórico:',[h['acao'] for h in historico])
print('Decisões persistiram; nenhum registro operacional foi criado.')

Histórico: ['registrar', 'corrigir', 'aceitar']
Decisões persistiram; nenhum registro operacional foi criado.


In [6]:
# LIMPEZA: os dados deste notebook são temporários; não modifica o banco de demonstração do terminal.
banco.fechar()
temporario.cleanup()
print('Notebook concluído sem API e sem efeitos operacionais.')

Notebook concluído sem API e sem efeitos operacionais.
